# calo-ddpm — Sweep analysis (box sizes, DDRM η, centralities, baselines)

Companion to `calo_method_comparison.ipynb` (which compares algorithms at a
FIXED geometry).  This notebook analyzes **parameter sweeps**: metrics as a
function of box size (cent0 vs cent4), the DDRM η sweep, with the
model-free baselines (`noise`, `meanfill`) as floors.  Runs are keyed by
(group, algorithm, box, η) — no collisions between sweep points.

All scalar metrics come from `calo_inpaint.analysis.summarize_run`
(progress-truncated, NaN-guarded, log-space value metrics by default).
Reference lines: pull std → the finite-J calibrated value
√((1+1/J)(J−1)/(J−3)) ≈ 1.031 at J=50, coverage → nominal, bias → 0.

In [ ]:
import os, sys
import numpy as np
import matplotlib.pyplot as plt
sys.path.insert(0, os.path.abspath('..'))

from calo_inpaint.analysis import collect_runs, pull_std_reference

ROOT = os.environ.get('CALO_ROOT', os.path.join('..', 'workdir'))

# group label -> glob pattern(s); edit to match your layout
GROUPS = {
    'cent0'      : f'{ROOT}/inpaint_study/*_box*_eta*_phi*_S*',
    'cent4'      : f'{ROOT}/inpaint_study_cent4/*_box*_eta*_phi*_S*',
    'ddrm_eta'   : f'{ROOT}/ddrm_eta_sweep/eta*/*_box*_eta*_phi*_S*',
}
SPACE = os.environ.get('CALO_SPACE', 'log')     # value-level metric space

ALG_COLORS = {'ddnm':'r', 'repaint':'g', 'ddrm':'b', 'mcg2':'y',
              'pigdm2':'m', 'noise':'c', 'meanfill':'0.4',
              'mcg':'y', 'pigdm':'m'}

summaries = {}
for label, pat in GROUPS.items():
    summaries[label] = collect_runs(pat, space=SPACE)
    print(f'{label}: {len(summaries[label])} runs')

## DDRM η sweep — spatial bias maps

Top row: mean(inpainted − truth) per tower, on a COMMON color scale across
η so panels are directly comparable.  Bottom row: the same bias divided by
its standard error (a z-map) — coherent structure beyond |z| ≈ 2–3 is
significant; the earlier finding was a spatially uniform positive bias at
the default η = 0.85.

In [ ]:
from calo_inpaint.analysis import load_run, CLIP_MIN

eta_rows = sorted([r for r in summaries.get('ddrm_eta', [])
                   if r['ddrm_eta'] is not None],
                  key=lambda r: r['ddrm_eta'])
if eta_rows:
    maps = []
    for r in eta_rows:
        d = load_run(r['rundir'])
        s, t = d['samples'], d['truth']
        if SPACE == 'log':
            s = np.log(np.clip(s, CLIP_MIN * 1e-3, None))
            t = np.log(np.clip(t, CLIP_MIN * 1e-3, None))
        diff = s.mean(axis=1) - t                      # (n, b, b)
        bias = diff.mean(axis=0)
        se   = diff.std(axis=0, ddof=1) / np.sqrt(diff.shape[0])
        maps.append((r['ddrm_eta'], bias, bias / np.clip(se, 1e-12, None)))

    vmax  = max(np.abs(m[1]).max() for m in maps)      # common scales
    zmax  = max(3.0, min(8.0, max(np.abs(m[2]).max() for m in maps)))
    unit  = '[ln GeV]' if SPACE == 'log' else '[GeV]'

    fig, axes = plt.subplots(2, len(maps), figsize=(3.4*len(maps), 6.2),
                             squeeze=False)
    for col, (eta, bias, z) in enumerate(maps):
        im0 = axes[0][col].imshow(bias, cmap='RdBu_r', vmin=-vmax, vmax=vmax)
        axes[0][col].set_title(f'η = {eta:g}', fontsize=10)
        im1 = axes[1][col].imshow(z, cmap='RdBu_r', vmin=-zmax, vmax=zmax)
        for row in (0, 1):
            axes[row][col].set_xticks([]); axes[row][col].set_yticks([])
        sign_frac = float((bias > 0).mean())
        axes[1][col].set_xlabel('phi index')
        axes[1][col].text(0.02, 0.02, f'sign frac = {sign_frac:.2f}',
                          transform=axes[1][col].transAxes, fontsize=7,
                          va='bottom', ha='left')
    fig.colorbar(im0, ax=axes[0], fraction=0.02, label=f'bias {unit}')
    fig.colorbar(im1, ax=axes[1], fraction=0.02, label='bias / SE')
    axes[0][0].set_ylabel('eta index')
    axes[1][0].set_ylabel('eta index')
    fig.suptitle(f'DDRM η sweep — spatial bias (box {eta_rows[0]["box"]}, '
                 f'{SPACE} space; common scales)')
    plt.show()
else:
    print('no DDRM eta-sweep runs found')

## Summary table (all runs)

In [ ]:
hdr = (f'{"group":10s} {"alg":9s} {"box":>3s} {"eta":>5s} {"n":>5s} '
       f'{"pull_std":>8s} {"(ref)":>6s} {"cov50":>6s} {"cov90":>6s} '
       f'{"sbc_p":>8s} {"bias":>8s} {"crps":>7s} {"fwd":>6s} {"rev":>6s}')
print(hdr); print('-' * len(hdr))
for label, rows in summaries.items():
    for r in rows:
        eta = f"{r['ddrm_eta']:.2f}" if r['ddrm_eta'] is not None else '  - '
        print(f"{label:10s} {r['algorithm']:9s} {r['box']:3d} {eta:>5s} "
              f"{r['n_images']:5d} {r['pull_std']:8.3f} "
              f"{r['pull_std_ref']:6.3f} {r['cov50']:6.3f} {r['cov90']:6.3f} "
              f"{r['sbc_p']:8.2g} {r['bias_mean']:8.4f} {r['crps']:7.4f} "
              f"{r['slope_fwd']:6.3f} {r['slope_rev']:6.3f}")

## Box-size sweep — calibration vs dead-region size

Each panel: metric vs box size, one line per (group, algorithm).
Expectations: calibration is hardest for large boxes (weaker
conditioning); `noise`/`meanfill` anchor the floor; cent0 vs cent4 tests
centrality-independence.

In [ ]:
def sweep_lines(metric, ylabel, ref=None, groups=('cent0', 'cent4')):
    plt.figure(figsize=(6.5, 4))
    for label in groups:
        rows = [r for r in summaries.get(label, [])
                if r['ddrm_eta'] in (None, 0.85)]     # exclude eta sweep pts
        by_alg = {}
        for r in rows:
            by_alg.setdefault(r['algorithm'], []).append(r)
        for alg, rs in sorted(by_alg.items()):
            rs = sorted(rs, key=lambda r: r['box'])
            if len(rs) < 1:
                continue
            ls = '-' if label == 'cent0' else '--'
            plt.plot([r['box'] for r in rs], [r[metric] for r in rs],
                     ls + 'o', ms=4, color=ALG_COLORS.get(alg, 'k'),
                     label=f'{alg} ({label})')
    if ref is not None:
        plt.axhline(ref, color='k', ls=':', lw=1, label='calibrated')
    plt.xlabel('box size [towers]'); plt.ylabel(ylabel)
    plt.grid(alpha=0.3); plt.legend(fontsize=7, ncol=2)
    plt.tight_layout(); plt.show()

J_ref = None
for rows in summaries.values():
    if rows:
        J_ref = rows[0]['n_samples']; break

sweep_lines('pull_std', 'pull std (log space)',
            ref=pull_std_reference(J_ref or 50))
sweep_lines('cov90', 'empirical coverage @ 90%', ref=0.90)
sweep_lines('cov50', 'empirical coverage @ 50%', ref=0.50)
sweep_lines('crps', 'mean CRPS (log space)')
sweep_lines('bias_mean', 'mean bias (log space)', ref=0.0)
sweep_lines('slope_rev', 'reverse slope (truth ~ posterior mean)', ref=1.0)

## DDRM η sweep

Calibration of the DDRM(η) family at fixed geometry.  Predictions on
record: the intrinsic variational distortion decreases as η→0 (analytic:
std ratio 1.00/0.995/0.96/0.86/0.70 at η = 0/0.2/0.5/0.85/1), while
deterministic-transport score-error accumulation grows as η→0 — a
U-shaped calibration curve is expected, with the published default
η = 0.85 marked.  The DDNM point at the same box is the ancestral-transport
reference.

In [ ]:
eta_rows = sorted([r for r in summaries.get('ddrm_eta', [])
                   if r['ddrm_eta'] is not None],
                  key=lambda r: r['ddrm_eta'])
if eta_rows:
    box_eta = eta_rows[0]['box']
    ddnm_ref = next((r for r in summaries.get('cent0', [])
                     if r['algorithm'] == 'ddnm' and r['box'] == box_eta),
                    None)
    etas = [r['ddrm_eta'] for r in eta_rows]

    fig, axes = plt.subplots(1, 4, figsize=(16, 3.6))
    panels = [('pull_std', 'pull std',
               pull_std_reference(eta_rows[0]['n_samples'])),
              ('cov90', 'coverage @ 90%', 0.90),
              ('bias_mean', 'mean bias (log)', 0.0),
              ('crps', 'CRPS (log)', None)]
    for ax, (metric, ylabel, ref) in zip(axes, panels):
        ax.plot(etas, [r[metric] for r in eta_rows], 'b-o', label='ddrm(η)')
        ax.axvline(0.85, color='b', ls=':', lw=1, label='published default')
        if ref is not None:
            ax.axhline(ref, color='k', ls=':', lw=1)
        if ddnm_ref is not None:
            ax.axhline(ddnm_ref[metric], color='r', ls='--', lw=1,
                       label=f'ddnm box{box_eta}')
        ax.set_xlabel('η'); ax.set_ylabel(ylabel); ax.grid(alpha=0.3)
    axes[0].legend(fontsize=7)
    fig.suptitle(f'DDRM η sweep (box {box_eta})')
    plt.tight_layout(); plt.show()
else:
    print('no DDRM eta-sweep runs found under', GROUPS['ddrm_eta'])

## Baselines as floors

Direct comparison at each box size: how much of the DDPM's advantage over
"fill with known-pixel statistics" survives per metric.  `meanfill` has
zero posterior width by construction — its coverage is ~0 and its pull std
diverges; it is meaningful only for CRPS/bias.

In [ ]:
rows0 = summaries.get('cent0', [])
boxes = sorted({r['box'] for r in rows0})
if boxes:
    print(f'{"box":>4s}  ' + ''.join(f'{a:>12s}' for a in
          ('ddnm', 'noise', 'meanfill')) + '   (CRPS, log space)')
    for b in boxes:
        vals = []
        for alg in ('ddnm', 'noise', 'meanfill'):
            r = next((r for r in rows0
                      if r['algorithm'] == alg and r['box'] == b), None)
            vals.append(f"{r['crps']:12.4f}" if r else f'{"-":>12s}')
        print(f'{b:4d}  ' + ''.join(vals))